In [1]:
from pathlib import Path
import sys
import importlib

# Locate the capsule/project root independently of the working directory.
# In Code Ocean this notebook lives in /code/<subfolder>/ and the data in /data.
# Locally it lives in <repo>/code/<subfolder>/ with the data in <repo>/data.
def _find_code_dir(start):
    for p in [start, *start.parents]:
        if (p / "utils.py").is_file():
            return p
        if (p / "code" / "utils.py").is_file():
            return p / "code"
    raise FileNotFoundError("could not find code/utils.py")

CODE_DIR = _find_code_dir(Path.cwd().resolve())
DATA_DIR = CODE_DIR.parent / "data"
# Add the code folder to sys.path so that `utils`, `statistic_checks` and
# `mot_analysis` can be imported
sys.path.append(str(CODE_DIR))

from utils import *
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List
import importlib

In [2]:
main_df = pd.read_csv(DATA_DIR / "search_performance_long.csv")

conditions = [(1,7),(8,15),(16,23),(25,31),(32,39),(40,47),(49,55),(56,63),(64,71),(72,77)]

def map_condition(trial_id):
    for idx, (start, end) in enumerate(conditions):
        if start <= trial_id <= end:
            return f"Cond_{idx}" # Or simply idx + 1
    return None

# 3. Apply the mapping to create the 'condition' column
main_df['condition'] = main_df['trial_id'].apply(map_condition)

main_df = main_df[main_df['condition'] != 'Cond_9']
main_df = main_df[main_df['condition'].isin(['Cond_3','Cond_4','Cond_5','Cond_6', 'Cond_7', 'Cond_8'])]
# main_df = main_df[main_df['condition'].isin(['Cond_0','Cond_1','Cond_2'])]

main_df = main_df.dropna()

In [ ]:
# --- Timing Based Aggregators (Seconds) ---

def replace_30_mean(x):
    arr = np.where(np.array(x) == -1, 30.0, x)
    return np.mean(arr)

def replace_30_median(x):
    arr = np.where(np.array(x) == -1, 30.0, x)
    return np.median(arr)

def IES_mean(x):
    arr = np.array(x)
    correct = arr[arr != -1]
    if len(correct) == 0: return np.nan 
    return np.mean(correct) / (len(correct) / len(arr))

def IES_median(x):
    arr = np.array(x)
    correct = arr[arr != -1]
    if len(correct) == 0: return np.nan
    return np.median(correct) / (len(correct) / len(arr))

# --- Log-Transformed Aggregators ---
# Note: We apply log(x) to the penalized or correct values before the mean/median

def log_replace_30_mean(x):
    arr = np.where(np.array(x) == -1, 30.0, x)
    return np.mean(np.log(arr))

def log_replace_30_median(x):
    arr = np.where(np.array(x) == -1, 30.0, x)
    return np.median(np.log(arr))

def log_IES_mean(x):
    """Log of the IES score (handles the ratio of mean RT and PC)"""
    val = IES_mean(x)
    return np.log(val) if not np.isnan(val) else np.nan

def log_IES_median(x):
    val = IES_median(x)
    return np.log(val) if not np.isnan(val) else np.nan

# --- Accuracy Based ---

def pc(x):
    arr = np.array(x)
    return len(arr[arr != -1]) / len(arr)



def custom(x):
    # x = x - 0.4
    arr = np.where(np.array(x) == -1, 30.0, x)
    return np.median(np.log(np.log((arr))))

# Final list

aggregators = [
    replace_30_mean, 
    replace_30_median,
    IES_mean, 
    IES_median,

]

log_aggregators = [
    log_replace_30_mean,
    log_replace_30_median,
    log_IES_mean,
    log_IES_median]

In [4]:
from statistic_checks import GLMs, MixedEffectStatistics
importlib.reload(GLMs)

results = GLMs.aggregators_pipeline(main_df, "reaction_time", log_aggregators)

results["multiplicative_coef"] = np.exp(results["group_coef"])
results

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


                       group_coef  group_p_value  marginal_r2  conditional_r2
aggregator                                                                   
log_replace_30_mean        0.2158         0.0057       0.8144          0.8711
log_replace_30_median      0.2729         0.0017       0.7890          0.8350
log_IES_mean               0.2289         0.0027       0.8540          0.8982
log_IES_median             0.2703         0.0020       0.7846          0.8322


,group_coef,group_p_value,marginal_r2,conditional_r2,multiplicative_coef
aggregator,,,,,
log_replace_30_mean,0.2158,0.0057,0.8144,0.8711,1.240854
log_replace_30_median,0.2729,0.0017,0.7890,0.8350,1.313769
log_IES_mean,0.2289,0.0027,0.8540,0.8982,1.257216
log_IES_median,0.2703,0.0020,0.7846,0.8322,1.310357
